In [ ]:
import requests, bs4, lxml, pandas
print("All libraries successfully imported!")


In [ ]:
import requests

# Step 1: Define the target URL
url = "https://towardsdatascience.com/"

# Step 2: Define a User-Agent header
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Step 3: Send the GET request
response = requests.get(url, headers=headers)

# Step 4: Check status code
print("--- Initiating Contact ---")
print(f"Status Code: {response.status_code}")

# Step 5: Preview HTML content
print("HTML Content Preview (First 500 characters):")
print(response.text[:500])

print("---")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Step 1: Get HTML content
url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)
response.raise_for_status()
html_content = response.text

# Step 2: Parse HTML
soup = BeautifulSoup(html_content, 'lxml')

# Step 3: Check title
print("--- Parsing HTML Content ---")

if soup.title:
    print(f"Parsed Page Title: {soup.title.text.strip()}")
else:
    print("Page title not found.")

print("---")


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')

print("--- PROCEDURE 6: Extract Data from HTML Tables ---")

target_table = soup.find('table', class_='wikitable')

if target_table:
    table_headers = [th.text.strip() for th in target_table.find_all('th')]

    table_data = []

    for row in target_table.find_all('tr')[1:6]:
        cells = row.find_all('td')
        cell_texts = [cell.text.strip() for cell in cells]
        table_data.append(cell_texts)

    print("\nFirst 5 Data Rows:")
    for i, row in enumerate(table_data):
        print(f"Row {i+1}: {row}")

    df = pd.DataFrame(table_data)
    print("\nDataFrame Preview:")
    print(df.head())

else:
    print("Table not found.")


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

print("--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---")

start_url = "http://quotes.toscrape.com/"
base_domain = urlparse(start_url).netloc

all_quotes = []
pages_to_collect = 3

current_page_url = start_url
page_counter = 0

while page_counter < pages_to_collect:
    print(f"\nFetching page {page_counter + 1}: {current_page_url}")

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}

        response = requests.get(current_page_url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'lxml')

        # Extract quotes
        quote_elements = soup.find_all('div', class_='quote')

        if quote_elements:
            for quote_item in quote_elements:
                text = quote_item.find('span', class_='text').text.strip()
                author = quote_item.find('small', class_='author').text.strip()

                all_quotes.append({
                    'text': text,
                    'author': author
                })

            print(f"Collected {len(quote_elements)} quotes from this page.")
        else:
            print("No quotes found. Stopping.")
            break

        # Find next page
        next_li = soup.find('li', class_='next')

        if next_li:
            next_link = next_li.find('a', href=True)

            if next_link:
                next_page_url = urljoin(start_url, next_link['href'])
                current_page_url = next_page_url
                page_counter += 1

                time.sleep(1)
            else:
                print("Next link not found. Stopping.")
                break
        else:
            print("No next page. Finished scraping.")
            break

    except requests.exceptions.RequestException as e:
        print(f"Network error: {e}")
        break

    except Exception as e:
        print(f"Unexpected error: {e}")
        break

print(f"\n--- Total Quotes Collected: {len(all_quotes)} ---")

for q in all_quotes[:5]:
    print(f"- \"{q['text']}\" — {q['author']}")

print("--- End of Procedure ---")


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

print("--- PROCEDURE 8: Clean and Standardize Extracted Data ---")

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    table = soup.find('table', class_='wikitable')

    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]

        software_idx = headers_text.index('Software') if 'Software' in headers_text else -1
        creator_idx = headers_text.index('Creator') if 'Creator' in headers_text else -1
        initial_release_idx = headers_text.index('Initial release') if 'Initial release' in headers_text else -1

        required_indices = [software_idx, creator_idx, initial_release_idx]

        if any(idx == -1 for idx in required_indices):
            print("ERROR: Missing required columns.")
            print("Found headers:", headers_text)

        else:
            cleaned_data_records = []

            data_rows = table.find_all('tr')[1:]

            for row_num, row in enumerate(data_rows):
                cols = row.find_all('td')

                if len(cols) > max(required_indices):
                    software_name = cols[software_idx].text.strip()
                    creator_name = cols[creator_idx].text.strip()
                    raw_initial_release = cols[initial_release_idx].text.strip()

                    clean_initial_release = re.sub(r'\[.*?\]', '', raw_initial_release).strip()

                    cleaned_data_records.append({
                        'Software': software_name,
                        'Creator': creator_name,
                        'Initial_Release_Date': clean_initial_release
                    })
                else:
                    print(f"Skipping row {row_num + 1}")

            print("\nCleaned Data (First 5 records):")
            for record in cleaned_data_records[:5]:
                print(record)

            df_cleaned = pd.DataFrame(cleaned_data_records)

            print("\nDataFrame Info:")
            df_cleaned.info()

            print("\nDataFrame Preview:")
            print(df_cleaned.head().to_string())

    else:
        print("Table not found on page.")

except requests.exceptions.RequestException as e:
    print(f"Network error: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")

finally:
    print("--- End of Procedure ---")


In [ ]:
import requests
from bs4 import BeautifulSoup

print("--- PROCEDURE 9: Implement Error Handling and Robustness ---")

test_scenarios = {
    "valid_article_page": "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "non_existent_article": "http://httpbin.org/status/404",
    "slow_server_sim": "http://httpbin.org/delay/6",
    "invalid_domain_connect": "http://this.domain.does.not.resolve.abc"
}

for scenario_name, test_url in test_scenarios.items():
    print(f"\nTesting Scenario: {scenario_name}")
    print(f"URL: {test_url}")

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}

        response = requests.get(test_url, headers=headers, timeout=5)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'lxml')

        if scenario_name == "valid_article_page":
            main_title_element = soup.find('h1', id='firstHeading')
        else:
            main_title_element = soup.find('h1')

        if main_title_element:
            print(f"SUCCESS: {main_title_element.text.strip()[:100]}")
        else:
            print("SUCCESS (but no title found)")

    except requests.exceptions.HTTPError as e:
        print(f"HTTP ERROR: {e.response.status_code} - {e.response.reason}")

    except requests.exceptions.ConnectionError:
        print("CONNECTION ERROR")

    except requests.exceptions.Timeout:
        print("TIMEOUT ERROR (5 seconds exceeded)")

    except requests.exceptions.RequestException as e:
        print(f"REQUEST ERROR: {e}")

    except Exception as e:
        print(f"UNEXPECTED ERROR: {e}")

print("--- END OF PROCEDURE ---")


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
import re
import sqlite3

print("\n--- PROCEDURE 10: Store the Extracted Data ---")

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    table = soup.find('table', class_='wikitable')

    if table:
        table_headers_raw = [th.text.strip() for th in table.find('tr').find_all('th')]

        table_headers = [
            re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip()
            for h in table_headers_raw
        ]

        data_rows = table.find_all('tr')[1:]

        final_data_to_save = []

        for row in data_rows[:15]:
            cols = row.find_all('td')
            row_data = {}

            for j, col in enumerate(cols):
                if j < len(table_headers):
                    row_data[table_headers[j]] = col.text.strip()

            if row_data:
                final_data_to_save.append(row_data)

        df_output = pd.DataFrame(final_data_to_save)

        print("\n--- Data Preview ---")
        print(df_output.head().to_string())

        # CSV
        csv_file = "deep_learning_data.csv"
        df_output.to_csv(csv_file, index=False)
        print(f"\nSaved CSV: {csv_file}")

        # JSON
        json_file = "deep_learning_data.json"
        df_output.to_json(json_file, orient="records", indent=4)
        print(f"Saved JSON: {json_file}")

        # SQLite
        db_file = "deep_learning_data.db"
        conn = sqlite3.connect(db_file)

        df_output.to_sql("dl_table", conn, if_exists="replace", index=False)
        conn.close()

        print(f"Saved SQLite DB: {db_file}")

        # Verify files
        if os.path.exists(csv_file) and os.path.exists(json_file) and os.path.exists(db_file):
            print("\nAll files successfully created ✔")
        else:
            print("\nFile verification failed ❌")

    else:
        print("Table not found on page.")

except requests.exceptions.RequestException as e:
    print(f"Network error: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")

print("--- END OF PROCEDURE ---")


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
import re
import sqlite3

print("\n--- PROCEDURE 10: Store the Extracted Data ---")

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    final_data_to_save = []

    table = soup.find('table', class_='wikitable')

    if table:
        table_headers_raw = [th.text.strip() for th in table.find('tr').find_all('th')]

        table_headers = [
            re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip()
            for h in table_headers_raw
        ]

        data_rows = table.find_all('tr')[1:]

        for row in data_rows[:15]:
            cols = row.find_all('td')
            row_data = {}

            for j, col in enumerate(cols):
                if j < len(table_headers):
                    header_key = table_headers[j]
                    row_data[header_key] = col.text.strip()

            if row_data:
                final_data_to_save.append(row_data)

        df_output = pd.DataFrame(final_data_to_save)

        print("\n--- Data Preview ---")
        print(df_output.head().to_string())

        # CSV
        csv_file = "deep_learning_software.csv"
        df_output.to_csv(csv_file, index=False)
        print(f"\nSaved CSV: {csv_file}")

        # JSON
        json_file = "deep_learning_software.json"
        df_output.to_json(json_file, orient="records", indent=4)
        print(f"Saved JSON: {json_file}")

        # SQLite
        db_file = "deep_learning_software.db"
        conn = sqlite3.connect(db_file)

        df_output.to_sql("dl_software", conn, if_exists="replace", index=False)
        conn.close()

        print(f"Saved SQLite DB: {db_file}")

        # Verification
        if os.path.exists(csv_file) and os.path.exists(json_file) and os.path.exists(db_file):
            print("\nAll files successfully created ✔")
        else:
            print("\nFile verification failed ❌")

    else:
        print("Table not found on page.")

except requests.exceptions.RequestException as e:
    print(f"Network error: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")

finally:
    print("--- END OF PROCEDURE ---")


In [ ]:
import requests

# Fetching HTML Status
url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    print(f"HTTP Status Code: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Parsing HTML and Getting Title Tag
url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    print(f"Page Title (from <title> tag): {soup.title.text}")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Extracting Main Article Heading
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    main_heading = soup.find('h1', id='firstHeading')

    if main_heading:
        print(f"Main Heading: '{main_heading.text.strip()}'")
    else:
        print("Main heading not found.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Extracting First Paragraph
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    content_div = soup.find('div', id='mw-content-text')

    first_paragraph = None

    if content_div:
        paragraphs = content_div.find_all('p')

        for p in paragraphs:
            text = p.text.strip()

            if text and not p.find_parent('table', class_='infobox'):
                first_paragraph = text
                break

    if first_paragraph:
        print(f"First Paragraph (truncated): '{first_paragraph[:150]}...'")
    else:
        print("First meaningful paragraph not found.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Extracting the first quote and author from Quotes To Scrape
url = "http://quotes.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    # Find first quote block
    first_quote_div = soup.find('div', class_='quote')

    if first_quote_div:
        quote_text_element = first_quote_div.find('span', class_='text')
        author_element = first_quote_div.find('small', class_='author')

        if quote_text_element and author_element:
            quote = quote_text_element.text.strip()
            author = author_element.text.strip()

            print(f"Sample Quote: '{quote}'")
            print(f"Sample Author: {author}")
        else:
            print("Quote or author element not found inside quote block.")
    else:
        print("No quote div found. Website structure may have changed.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Extracting ALL Quote Authors from Quotes To Scrape
url = "http://quotes.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    all_quote_divs = soup.find_all('div', class_='quote')

    if all_quote_divs:
        print("Authors found on the page:")

        authors_list = []

        for i, quote_div in enumerate(all_quote_divs):
            author_element = quote_div.find('small', class_='author')

            if author_element:
                author_name = author_element.text.strip()
                authors_list.append(author_name)
                print(f"- {author_name}")
            else:
                print(f"- Author not found for quote #{i+1}")

        print(f"\nTotal unique authors found: {len(set(authors_list))}")

    else:
        print("No quote containers found on the page.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# Raw Numerical String with Commas
url = "https://en.wikipedia.org/wiki/List_of_countries_by_population"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    table = soup.find('table', class_='wikitable')

    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]

        population_idx = -1

        for i, header in enumerate(headers_text):
            if 'Population' in header:
                population_idx = i
                break

        if population_idx != -1:
            rows = table.find_all('tr')

            if len(rows) > 1:
                first_data_row = rows[1]
                cols = first_data_row.find_all('td')

                if len(cols) > population_idx:
                    raw_population = cols[population_idx].text.strip()
                    print(f"Raw Population String: '{raw_population}'")
                else:
                    print("Population column index out of range in row.")
            else:
                print("No data rows found.")

        else:
            print("Population column not found in headers.")

    else:
        print("Table not found on page.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")


In [ ]:
import pandas as pd

# Save Laptops Data to CSV

items = soup.select(".thumbnail")

data = []

for it in items:
    title_elem = it.select_one(".title")
    price_elem = it.select_one(".price")

    if title_elem and price_elem:
        data.append({
            "name": title_elem.get("title", "").strip(),
            "price": price_elem.text.strip()
        })

df = pd.DataFrame(data)

df.to_csv("laptops.csv", index=False)

print("Saved laptops.csv")


In [ ]:
import requests

# Handling Timeout Error
url = "http://httpbin.org/delay/6"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()

    print("SUCCESS: Page fetched (unexpected for timeout).")

except requests.exceptions.Timeout as e:
    print("ERROR (Timeout): Request to", url, "timed out after 5 seconds.")
    print(f"Details: {e}")

except requests.exceptions.RequestException as e:
    print(f"ERROR (General Request): {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

# Storing Data to CSV

url = "http://quotes.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')

all_quote_divs = soup.find_all('div', class_='quote')

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find('span', class_='text')
    author_element = quote_div.find('small', class_='author')

    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"

    scraped_data.append({
        'Quote': quote,
        'Author': author
    })

df_scraped = pd.DataFrame(scraped_data)

csv_file_path = 'web_scraped_quotes.csv'
df_scraped.to_csv(csv_file_path, index=False, encoding='utf-8')

print(f"Saved file: {csv_file_path}")


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import os

# Storing Web Scraped Data to JSON

url = "http://quotes.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')

all_quote_divs = soup.find_all('div', class_='quote')

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find('span', class_='text')
    author_element = quote_div.find('small', class_='author')

    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"

    scraped_data.append({
        'quote': quote,
        'author': author
    })

json_file_path = 'web_scraped_quotes.json'

with open(json_file_path, 'w', encoding='utf-8') as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)

print(f"Saved file: {json_file_path}")


In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import os

# Storing Web Scraped Data to SQLite

url = "http://quotes.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')

all_quote_divs = soup.find_all('div', class_='quote')

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find('span', class_='text')
    author_element = quote_div.find('small', class_='author')

    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"

    scraped_data.append({
        'quote': quote,
        'author': author
    })

# --- SQLite Storage ---
db_file_path = 'web_scraped_quotes.db'

conn = sqlite3.connect(db_file_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS quotes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    quote TEXT NOT NULL,
    author TEXT NOT NULL
)
""")

for item in scraped_data:
    cursor.execute(
        "INSERT INTO quotes (quote, author) VALUES (?, ?)",
        (item['quote'], item['author'])
    )

conn.commit()
conn.close()

print(f"Saved database: {db_file_path}")


In [ ]:
import pandas as pd
import sqlite3
import os

# Verifying Scraped Data in SQLite

db_file_path = 'web_scraped_quotes.db'
table_name = 'quotes'

# Check if database exists
if os.path.exists(db_file_path):

    # Connect to SQLite database
    conn = sqlite3.connect(db_file_path)

    # Read data into DataFrame
    query = f"SELECT * FROM {table_name}"
    df_read = pd.read_sql_query(query, conn)

    # Display results
    print("\nData read from SQLite ('web_scraped_quotes.db'):")
    print(df_read.to_string())

    # Close connection
    conn.close()

else:
    print(f"SQLite database '{db_file_path}' not found. Please run the scraper first.")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Extracting Quotes from First Page

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    quote_elements = soup.find_all('div', class_='quote')

    if quote_elements:
        print(f"Found {len(quote_elements)} quotes on the first page.\n")

        for i, quote_item in enumerate(quote_elements[:2]):
            text = quote_item.find('span', class_='text').text.strip()
            author = quote_item.find('small', class_='author').text.strip()

            print(f"Quote {i+1}: \"{text[:70]}...\" - {author}")
    else:
        print("No quotes found on the first page.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Locating 'Next Page' Link

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    next_li = soup.find('li', class_='next')

    if next_li:
        next_link_element = next_li.find('a', href=True)

        if next_link_element:
            next_page_relative_url = next_link_element['href']
            full_next_page_url = urljoin(url, next_page_relative_url)

            print(f"Found 'Next Page' link: {full_next_page_url}")
        else:
            print("Next page <a> tag not found within 'next' li.")
    else:
        print("No 'next' pagination li found (likely last page).")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup

# Extracting Raw Table Headers

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    table = soup.find('table', class_='wikitable')

    if table:
        header_cells = table.find('tr').find_all('th')
        raw_headers = [th.text.strip() for th in header_cells]

        print(f"Raw Headers: {raw_headers}")
    else:
        print("Table not found.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# Standardizing Table Headers

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {
    "User-Agent": "Mozilla/5.0"
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    table = soup.find("table", class_="wikitable")

    if not table:
        print("Table not found.")
    else:
        header_cells = table.find("tr").find_all("th")

        raw_headers = [cell.get_text(strip=True) for cell in header_cells]

        print("Raw Headers from Web Scraping:")
        print(raw_headers)

        standardized_headers = [
            re.sub(r"\[.*?\]|\s\(.*?\)", "", h)   # remove [a], (2024), etc.
            .replace(" ", "_")                   # spaces → underscore
            .strip()
            for h in raw_headers
        ]

        print("\nStandardized Headers:")
        print(standardized_headers)

except requests.exceptions.RequestException as e:
    print(f"Request error: {e}")


In [ ]:
ERROR (Connection): Could not establish connection to http://this.domain.does.not.resolve.abc. Details: HTTPConnectionPool(host='this.domain.does.not.resolv e.abc', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnect ion object at 0x00000207B2AA3DF0>: Failed to resolve 'this.domain.does.not.resolve.abc' ([Errno 11001]
getaddrinfo failed)"))



In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# Extracting Wikipedia Last Modified Date

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')

    last_modified_element = soup.find('li', id='footer-info-lastmod')

    if last_modified_element:
        raw_text = last_modified_element.text.strip()

        date_match = re.search(r'on (\d{1,2} \w+ \d{4})', raw_text)

        if date_match:
            clean_date = date_match.group(1)
            print(f"Last Modified Date: '{clean_date}'")
        else:
            print(f"Last Modified Date (raw, unable to parse): '{raw_text}'")
    else:
        print("Last modified element not found.")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from datetime import datetime

base_url = "https://example.com/weather/"  # replace with actual timeanddate city URL
city = input("Enter city name: ")
days_input = input("Enter number of days (default 3): ")

days = int(days_input) if days_input.isdigit() else 3

file_name = "weather_data.csv"

# Create file + header if not exists
try:
    with open(file_name, "x", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp", "city", "forecast_date", "temperature",
            "temperature_min", "temperature_max", "condition",
            "wind", "humidity", "pressure", "visibility",
            "dew_point", "feels_like"
        ])
except FileExistsError:
    pass

for i in range(days):
    print(f"Scraping day {i+1}...")

    # Simulated scraping (replace with real parsing later)
    data = {
        "timestamp": datetime.now(),
        "city": city,
        "forecast_date": f"Day {i+1}",
        "temperature": "30°C",
        "temperature_min": "27°C",
        "temperature_max": "33°C",
        "condition": "Light rain",
        "wind": "10 km/h NE",
        "humidity": "80%",
        "pressure": "1012 hPa",
        "visibility": "10 km",
        "dew_point": "24°C",
        "feels_like": "34°C"
    }

    with open(file_name, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(data.values())

    time.sleep(2)

print("Weather logging complete. Saved to weather_data.csv")


In [ ]:
import json
import time
from datetime import datetime

city = input("Enter city name: ")
days_input = input("Number of forecast days (default 3): ")

days = int(days_input) if days_input.isdigit() else 3

file_name = "weather_data.json"

try:
    with open(file_name, "r", encoding="utf-8") as f:
        data_list = json.load(f)
except:
    data_list = []

for i in range(days):
    record = {
        "timestamp": str(datetime.now()),
        "city": city,
        "forecast_date": f"Day {i+1}",
        "temperature": "30°C",
        "temperature_min": "27°C",
        "temperature_max": "33°C",
        "condition": "Cloudy",
        "wind": "8 km/h",
        "humidity": "78%",
        "pressure": "1011 hPa",
        "visibility": "10 km",
        "dew_point": "23°C",
        "feels_like": "33°C"
    }

    data_list.append(record)
    time.sleep(1)

with open(file_name, "w", encoding="utf-8") as f:
    json.dump(data_list, f, indent=4)

print("Data saved to weather_data.json")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

city = input("Enter city name: ")

# Sample data (replace later with CSV if needed)
df = pd.DataFrame({
    "day": [1, 2, 3, 4, 5],
    "temp_min": [27, 26, 28, 27, 26],
    "temp_max": [33, 34, 32, 33, 35],
    "humidity": [80, 78, 85, 82, 79]  # optional annotation data
})

fig, ax = plt.subplots()

ax.set_xlim(1, 5)
ax.set_ylim(20, 40)

# Lines + POINTS (markers added)
line_min, = ax.plot([], [], marker="o", label="Min Temp")
line_max, = ax.plot([], [], marker="o", label="Max Temp")

ax.set_xlabel("Day")
ax.set_ylabel("Temperature (°C)")
ax.set_title(f"Weather Forecast Trend - {city}")
ax.legend()

# annotation text holder
annot = ax.text(0.02, 0.95, "", transform=ax.transAxes)

def update(frame):
    x = df["day"][:frame]
    y_min = df["temp_min"][:frame]
    y_max = df["temp_max"][:frame]

    line_min.set_data(x, y_min)
    line_max.set_data(x, y_max)

    # show last point info
    if frame > 0:
        annot.set_text(
            f"Humidity: {df['humidity'][frame-1]}%"
        )

    return line_min, line_max, annot

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(df) + 1,
    interval=800,
    blit=False
)

# ✅ SAVE AS GIF (THIS IS THE IMPORTANT PART)
ani.save("forecast_plot.gif", writer="pillow", fps=2)

print("Saved as forecast_plot.gif")
plt.show()


